In [21]:
%load_ext autoreload
%autoreload 2

In [43]:
import pandas as pd
from evalviz.config import load_runs_from_files, load_methods, load_benchmarks
from evalviz.io import load_results_from_runs
from evalviz.preprocess import apply_mappings, add_dataset_type_columns, apply_dataset_merge_rules
from evalviz.enrich import attach_method_metadata, attach_benchmark_metadata
from evalviz.capabilities import (
    annotate_capabilities_hierarchical,
    get_capability_ordering,
)
from evalviz.benchmarks import build_subset_capability_lookup
import tueplots
from tueplots import figsizes
import re

ours = "configs/runs_ours_final.json"
# ours = "configs/runs_openclip.json"

runs = load_runs_from_files(
    "configs/runs_baselines.json",
    "configs/runs_external.json",
    ours
)

methods = load_methods("configs/methods.json")
bench_cfg = load_benchmarks("configs/benchmarks.json")

df = load_results_from_runs(runs)
df = apply_dataset_merge_rules(df, bench_cfg)
df = apply_mappings(df, bench_cfg)
df = add_dataset_type_columns(df)
df = attach_method_metadata(df, methods)
df = attach_benchmark_metadata(df, bench_cfg)

# Build capability lookup and annotate with hierarchical capabilities
subset_cap_lookup = build_subset_capability_lookup(bench_cfg)
df = annotate_capabilities_hierarchical(df, subset_cap_lookup, bench_cfg)

# Filter the datasets with the run label
# Get only NegCLIP CS-CLIP ReadCLIP FSCCLIP Degla CLIP
# pattern = re.compile(r"^(NegCLIP|CS-CLIP|READCLIP|FSC CLIP|DeGLA|CLIP|CE CLIP|CLIC).*")
# df = df[df["run_label"].str.match(pattern)]
# Get canonical ordering for tables/plots
TOP_CAPS, SUB_CAPS = get_capability_ordering(bench_cfg)
print(f"Top-level capabilities: {TOP_CAPS}")
print(f"Sub-level capabilities: {SUB_CAPS}")

# Show sample of capability columns
df[["run_label", "dataset", "subset", "capability_top", "capability_sub"]].drop_duplicates().head(10)

Top-level capabilities: ['EntityContent', 'RelationalStructure', 'Binding', 'Linguistic']
Sub-level capabilities: ['ObjectRecognition', 'AttributeRecognition', 'CountingQuantity', 'ExistencePresence', 'PredicateSensitivity', 'RoleSensitivity', 'AttributeBinding', 'WordOrderSyntax', 'Coreference', 'Negation']


,run_label,dataset,subset,capability_top,capability_sub
692,NegCLIP (COCO),ARO,VG_Relation,RelationalStructure,PredicateSensitivity
3126,FSC CLIP (LAION+COCO),VL_CheckList,obj_location,EntityContent,ObjectRecognition
4575,READCLIP,ARO,VG_Attribution,Binding,AttributeBinding
6456,DAC (SAM),ColorFoil,all,EntityContent,AttributeRecognition
5988,DAC (LLM),ColorFoil,all,EntityContent,AttributeRecognition
1190,CE CLIP,ColorFoil,all,EntityContent,AttributeRecognition
3208,FSC CLIP (LAION+COCO),ARO,VG_Relation,RelationalStructure,PredicateSensitivity
6972,LaCLIP (CC12M),ARO,VG_Relation,RelationalStructure,PredicateSensitivity
8265,CS-CLIP,VL_CheckList,obj_location,EntityContent,ObjectRecognition
2224,FSC CLIP (CC3M),ColorFoil,all,EntityContent,AttributeRecognition


In [44]:
import matplotlib.pyplot as plt
plt.rcParams.update(
    figsizes.icml2024_full()
)

## 1. Per-Dataset Tables (Detailed + Compact)

In [45]:
from evalviz.tables import (
    build_compositional_tables_for_dataset,
    build_all_compositional_tables,
    make_latex_from_compositional_table,
    make_latex_summary_table,
    make_latex_capability_table_models_rows,
)

# Build all compositional tables (detailed + compact) for each dataset
all_tables = build_all_compositional_tables(df)

# Show available datasets
print("Available datasets:", list(all_tables.keys()))

Available datasets: ['ARO', 'BLA', 'COCO-CF', 'COLA', 'ColorFoil', 'ColorSwap', 'ControlledImages', 'MMVP', 'NegBench', 'SPEC', 'SugarCrepe', 'SugarCrepe++', 'VALSE', 'VL_CheckList', 'VisMin', 'Winoground']


# Generate LaTeX tables for each dataset (detailed and compact)
all_latex = {}

for ds_name, tables in all_tables.items():
    detailed = tables["detailed"]
    compact = tables["compact"]
    
    # Detailed table (all subsets)
    if not detailed.empty:
        latex_det = make_latex_from_compositional_table(
            detailed, df, ds_name,
            caption=f"{ds_name} detailed results (all subsets)",
            label=f"tab:{ds_name.lower().replace(' ', '_')}_detailed",
        )
        all_latex[f"{ds_name}_detailed"] = latex_det
    
    # Compact table (grouped subsets, if available)
    if compact is not None and not compact.empty:
        latex_cmp = make_latex_from_compositional_table(
            compact, df, ds_name,
            caption=f"{ds_name} compact results (grouped subsets)",
            label=f"tab:{ds_name.lower().replace(' ', '_')}_compact",
        )
        all_latex[f"{ds_name}_compact"] = latex_cmp

print(f"Generated {len(all_latex)} LaTeX tables")
print("Tables:", list(all_latex.keys()))

## 2. Summary Table (Dataset Averages)

In [46]:
# Summary table: each dataset averaged, rows = models
latex_summary = make_latex_summary_table(
    df,
    caption="Summary of compositional benchmark results. Each dataset averaged over all subsets.",
    label="tab:summary_compositional",
    include_overall=True,
    max_cols_per_table=16
)
print(latex_summary)

\begin{table}[t]
  \centering
  \scriptsize
  \caption{Summary of compositional benchmark results. Each dataset averaged over all subsets.}
  \label{tab:summary_compositional}
  \begin{adjustbox}{max width=\textwidth}
  \setlength{\tabcolsep}{2pt}
  \begin{tabular}{lccccccccccccccccc}
    \toprule
    Model & ARO & BLA & COCO-CF & COLA & ColorFoil & ColorSwap & ControlledImages & MMVP & NegBench & SPEC & SugarCrepe & SugarCrepe++ & VALSE & VL\_CheckList & VisMin & Winoground & Avg \\
    \midrule
    CLIP & \underline{48.5} & \underline{49.4} & \underline{74.0} & \underline{41.9} & \underline{84.2} & \underline{60.7} & \underline{41.4} & \underline{13.3} & \underline{30.6} & \underline{32.7} & \underline{76.8} & \underline{59.7} & \underline{65.4} & \underline{67.1} & \underline{57.9} & \underline{29.8} & \underline{52.1} \\
    \midrule
    LaCLIP (CC12M) & 39.4 & 50.2 & 41.0 & 21.9 & 69.1 & 56.7 & 41.8 & \textbf{15.6} & 25.5 & 29.2 & 63.5 & 46.0 & 57.3 & 67.6 & 33.7 & 26.2 & 42.8 \\


In [36]:
## 3. Capability Tables (Top-Level for Main Paper)

# Top-level capability table (for main paper)
latex_caps_top = make_latex_capability_table_models_rows(
    df, 
    level="top",
    caption="Performance by top-level capability category.",
    label="tab:capability_top",
)
print(latex_caps_top)

In [37]:
## 4. Capability Tables (Sub-Level for Appendix)

In [38]:
# Sub-level capability table (for appendix)
latex_caps_sub = make_latex_capability_table_models_rows(
    df, 
    level="sub",
    caption="Performance by sub-level capability category.",
    label="tab:capability_sub",
)
print(latex_caps_sub)

\begin{table}[t]
  \centering
  \scriptsize
  \caption{Performance by sub-level capability category.}
  \label{tab:capability_sub}
  \begin{adjustbox}{max width=\textwidth}
  \setlength{\tabcolsep}{2pt}
  \begin{tabular}{p{5.0cm}ccccccccccc}
    \toprule
    Model & AttributeBinding & AttributeRecognition & Coreference & CountingQuantity & ExistencePresence & Negation & ObjectRecognition & PredicateSensitivity & RoleSensitivity & WordOrderSyntax & Avg \\
    \midrule
    CLIP & \underline{39.3} & \underline{53.4} & \underline{51.4} & \underline{34.3} & \underline{24.1} & \underline{30.6} & \underline{65.0} & \underline{42.4} & \underline{34.6} & \underline{40.9} & \underline{41.6} \\
    \midrule
    LaCLIP (CC12M) & 37.7 & 47.5 & 52.2 & 29.6 & 29.0 & 25.5 & 55.3 & 41.1 & 35.3 & 38.1 & 39.1 \\
    TripletCLIP (CC12M) & 35.5 & 48.4 & 50.9 & 26.9 & 24.1 & 27.3 & 56.6 & 42.7 & 33.9 & 35.3 & 38.2 \\
    \midrule
    CE CLIP & 37.8 & 56.2 & \textbf{60.6} & 37.0 & 30.1 & 35.7 & 66.2 & 46.4 &

## 5. Hierarchical Capability Table (Category / Sub-Category with Multicolumn)

In [47]:
def make_latex_hierarchical_capability_table(
    df, bench_cfg, metric="text_contrastive_accuracy",
    caption="Performance by capability category and sub-category.",
    label="tab:capability_hierarchical",
    decimals=1,
    font_size="scriptsize",
    tabcolsep="3pt"
):
    """
    Create a LaTeX table with multicolumn headers showing:
    - Top row: Category names spanning their sub-categories
    - Second row: Sub-category names
    - Body: Model scores for each sub-category + category averages
    """
    from evalviz.tables import build_capability_table, _run_info_from_df, order_rows_by_groups, latex_escape
    
    # Get sub-level capability table
    pivot_sub, meta = build_capability_table(df, metric=metric, level="sub")
    if pivot_sub.empty:
        return "% Empty hierarchical capability table.\n"
    
    # Transpose to get models as rows
    tab = pivot_sub.T  # index=run_label, columns=sub-capability
    
    # Get hierarchy from config
    sub_buckets = bench_cfg.get("capability_schema", {}).get("sub_buckets", {})
    top_order = bench_cfg.get("capability_schema", {}).get("top_buckets", list(sub_buckets.keys()))
    
    # Build ordered list of (category, [sub-categories])
    hierarchy = []
    for cat in top_order:
        subs = sub_buckets.get(cat, [])
        # Only include subs that exist in our data
        valid_subs = [s for s in subs if s in tab.columns]
        if valid_subs:
            hierarchy.append((cat, valid_subs))
    
    # Order rows by method groups
    run_info = meta.fillna(False).astype(bool).to_dict(orient="index")
    methods = list(tab.index)
    ordered_methods, split_idxs = order_rows_by_groups(methods, run_info)
    tab = tab.loc[ordered_methods]
    
    # Helper functions
    def fmt(v):
        if pd.isna(v):
            return "--"
        return f"{v * 100:.{decimals}f}"
    
    def style_cell(s, method, is_best):
        info = run_info.get(method, {})
        if info.get("is_baseline", False):
            s = r"\underline{" + s + "}"
        if info.get("is_ours", False):
            s = r"\emph{" + s + "}"
        if is_best:
            s = r"\textbf{" + s + "}"
        return s
    
    # Calculate category averages
    cat_avgs = {}
    for cat, subs in hierarchy:
        cat_avgs[cat] = tab[subs].mean(axis=1)
    
    # Overall average
    all_subs = [s for _, subs in hierarchy for s in subs]
    overall_avg = tab[all_subs].mean(axis=1)
    
    # Find best per column
    best_per_sub = {s: tab[s].idxmax() for s in all_subs}
    best_per_cat = {cat: avg.idxmax() for cat, avg in cat_avgs.items()}
    best_overall = overall_avg.idxmax()
    
    # Build column spec
    # Model | sub1 sub2 | cat_avg | sub3 sub4 sub5 | cat_avg | ... | Overall
    col_parts = ["l"]  # Model column
    for cat, subs in hierarchy:
        col_parts.append("|")
        col_parts.append("c" * len(subs))  # sub-categories
        col_parts.append("c")  # category average
    col_parts.append("|c")  # Overall
    col_spec = "".join(col_parts)
    
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(rf"  \{font_size}")
    lines.append(f"  \\caption{{{caption}}}")
    lines.append(f"  \\label{{{label}}}")
    lines.append(r"  \begin{adjustbox}{max width=\textwidth}")
    lines.append(rf"  \setlength{{\tabcolsep}}{{{tabcolsep}}}")
    lines.append(rf"  \begin{{tabular}}{{{col_spec}}}")
    lines.append(r"    \toprule")
    
    # Header row 1: Category names with multicolumn
    header1 = [""]
    for cat, subs in hierarchy:
        n_cols = len(subs) + 1  # subs + category avg
        header1.append(rf"\multicolumn{{{n_cols}}}{{c}}{{{latex_escape(cat)}}}")
    header1.append("")
    lines.append("    " + " & ".join(header1) + r" \\")
    
    # Header row 2: Sub-category names + Avg for each category + Overall
    header2 = ["Model"]
    for cat, subs in hierarchy:
        for s in subs:
            header2.append(latex_escape(s))
        header2.append("Avg")
    header2.append("Overall")
    lines.append("    " + " & ".join(header2) + r" \\")
    lines.append(r"    \midrule")
    
    # Body rows
    for i, method in enumerate(tab.index):
        if i in split_idxs:
            lines.append(r"    \midrule")
        
        row = [latex_escape(method)]
        
        for cat, subs in hierarchy:
            # Sub-category scores
            for s in subs:
                val = fmt(tab.loc[method, s])
                val = style_cell(val, method, method == best_per_sub[s])
                row.append(val)
            
            # Category average
            cat_val = fmt(cat_avgs[cat].loc[method])
            cat_val = style_cell(cat_val, method, method == best_per_cat[cat])
            row.append(cat_val)
        
        # Overall average
        ov_val = fmt(overall_avg.loc[method])
        ov_val = style_cell(ov_val, method, method == best_overall)
        row.append(ov_val)
        
        lines.append("    " + " & ".join(row) + r" \\")
    
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"  \end{adjustbox}")
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

# Generate hierarchical capability table
latex_hierarchical = make_latex_hierarchical_capability_table(
    df, bench_cfg,
    caption="Performance by capability category and sub-category. Best per column in \\textbf{bold}, baseline \\underline{underlined}, our models \\emph{italic}.",
    label="tab:capability_hierarchical",
)
print(latex_hierarchical)

\begin{table}[t]
  \centering
  \scriptsize
  \caption{Performance by capability category and sub-category. Best per column in \textbf{bold}, baseline \underline{underlined}, our models \emph{italic}.}
  \label{tab:capability_hierarchical}
  \begin{adjustbox}{max width=\textwidth}
  \setlength{\tabcolsep}{3pt}
  \begin{tabular}{l|ccccc|ccc|cc|cccc|c}
    \toprule
     & \multicolumn{5}{c}{EntityContent} & \multicolumn{3}{c}{RelationalStructure} & \multicolumn{2}{c}{Binding} & \multicolumn{4}{c}{Linguistic} &  \\
    Model & ObjectRecognition & AttributeRecognition & CountingQuantity & ExistencePresence & Avg & PredicateSensitivity & RoleSensitivity & Avg & AttributeBinding & Avg & WordOrderSyntax & Coreference & Negation & Avg & Overall \\
    \midrule
    CLIP & \underline{65.0} & \underline{53.4} & \underline{34.3} & \underline{24.1} & \underline{44.2} & \underline{42.4} & \underline{34.6} & \underline{38.5} & \underline{39.3} & \underline{39.3} & \underline{40.9} & \underline{51.4} 

## 6. Bidirectional Table (I2T / T2I / Group Accuracy)

In [48]:
# First, let's check which datasets have image_contrastive_accuracy
metrics_by_dataset = df.groupby("dataset")["metric"].unique().to_dict()
print("Datasets with their available metrics:")
for ds, metrics in metrics_by_dataset.items():
    if "image_contrastive_accuracy" in metrics:
        print(f"  {ds}: {list(metrics)}")

Datasets with their available metrics:
  COCO-CF: ['text_contrastive_accuracy', 'image_contrastive_accuracy', 'group_contrastive_accuracy']
  COLA: ['multi_object_accuracy', 'text_contrastive_accuracy', 'image_contrastive_accuracy', 'group_contrastive_accuracy']
  ColorSwap: ['text_contrastive_accuracy_dalle3', 'image_contrastive_accuracy_dalle3', 'group_contrastive_accuracy_dalle3', 'count_dalle3', 'text_contrastive_accuracy_sdxl', 'image_contrastive_accuracy_sdxl', 'group_contrastive_accuracy_sdxl', 'count_sdxl', 'group_contrastive_accuracy_midjourney', 'count_midjourney', 'image_contrastive_accuracy_midjourney', 'text_contrastive_accuracy_midjourney', 'text_contrastive_accuracy', 'image_contrastive_accuracy', 'group_contrastive_accuracy', 'macro_contrastive_accuracy']
  MMVP: ['num_correct_pairs_text', 'num_correct_pairs_image', 'num_correct_pairs_group', 'structural_character_text_contrastive_accuracy', 'structural_character_image_contrastive_accuracy', 'structural_character_group_

In [49]:
def make_latex_bidirectional_table(
    df, 
    caption="Bidirectional performance: I2T (text contrastive), T2I (image contrastive), and Group accuracy.",
    label="tab:bidirectional",
    decimals=1,
    font_size="scriptsize",
    tabcolsep="3pt"
):
    """
    Create a LaTeX table with multicolumn headers showing:
    - For each dataset: I2T | T2I | Group columns
    - Rows = models
    """
    from evalviz.tables import _run_info_from_df, order_rows_by_groups, latex_escape, _ensure_method_flags
    
    df = _ensure_method_flags(df)
    
    # Filter to datasets that have image_contrastive_accuracy
    datasets_with_i2t_t2i = []
    for ds in df["dataset"].unique():
        ds_metrics = df[df["dataset"] == ds]["metric"].unique()
        if "image_contrastive_accuracy" in ds_metrics and "text_contrastive_accuracy" in ds_metrics:
            datasets_with_i2t_t2i.append(ds)
    
    datasets_with_i2t_t2i = sorted(datasets_with_i2t_t2i)
    
    if not datasets_with_i2t_t2i:
        return "% No datasets with both I2T and T2I metrics.\n"
    
    print(f"Datasets with I2T/T2I: {datasets_with_i2t_t2i}")
    
    # Build pivot tables for each metric
    def build_pivot(metric_name):
        d = df[(df["benchmark_type"] == "compositional") & (df["metric"] == metric_name)].copy()
        if d.empty:
            return pd.DataFrame()
        # Average over subsets and seeds
        agg = d.groupby(["dataset", "run_label"], as_index=False)["value"].mean()
        return agg.pivot_table(index="run_label", columns="dataset", values="value")
    
    pivot_i2t = build_pivot("text_contrastive_accuracy")
    pivot_t2i = build_pivot("image_contrastive_accuracy")
    pivot_grp = build_pivot("group_contrastive_accuracy")
    
    # Get common models across all pivots
    all_models = set(pivot_i2t.index) & set(pivot_t2i.index)
    if pivot_grp is not None and not pivot_grp.empty:
        all_models = all_models & set(pivot_grp.index)
        has_group = True
    else:
        has_group = False
    
    all_models = sorted(all_models)
    
    # Filter datasets to those in our pivots
    valid_datasets = [ds for ds in datasets_with_i2t_t2i 
                      if ds in pivot_i2t.columns and ds in pivot_t2i.columns]
    
    # Get run info for ordering
    run_info = _run_info_from_df(df)
    ordered_methods, split_idxs = order_rows_by_groups(all_models, run_info)
    
    # Helper
    def fmt(v):
        if pd.isna(v):
            return "--"
        return f"{v * 100:.{decimals}f}"
    
    def style_cell(s, method, is_best):
        info = run_info.get(method, {})
        if info.get("is_baseline", False):
            s = r"\underline{" + s + "}"
        if info.get("is_ours", False):
            s = r"\emph{" + s + "}"
        if is_best:
            s = r"\textbf{" + s + "}"
        return s
    
    # Find best per column
    best_i2t = {ds: pivot_i2t[ds].loc[ordered_methods].idxmax() for ds in valid_datasets if ds in pivot_i2t.columns}
    best_t2i = {ds: pivot_t2i[ds].loc[ordered_methods].idxmax() for ds in valid_datasets if ds in pivot_t2i.columns}
    best_grp = {}
    if has_group:
        best_grp = {ds: pivot_grp[ds].loc[ordered_methods].idxmax() for ds in valid_datasets if ds in pivot_grp.columns}
    
    # Column spec: Model | (I2T T2I [Grp]) per dataset | Avg
    n_metrics = 3 if has_group else 2
    col_parts = ["l"]
    for ds in valid_datasets:
        col_parts.append("|")
        col_parts.append("c" * n_metrics)
    col_parts.append("|ccc" if has_group else "|cc")  # Average columns
    col_spec = "".join(col_parts)
    
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(rf"  \{font_size}")
    lines.append(f"  \\caption{{{caption}}}")
    lines.append(f"  \\label{{{label}}}")
    lines.append(r"  \begin{adjustbox}{max width=\textwidth}")
    lines.append(rf"  \setlength{{\tabcolsep}}{{{tabcolsep}}}")
    lines.append(rf"  \begin{{tabular}}{{{col_spec}}}")
    lines.append(r"    \toprule")
    
    # Header row 1: Dataset names with multicolumn
    header1 = [""]
    for ds in valid_datasets:
        header1.append(rf"\multicolumn{{{n_metrics}}}{{c}}{{{latex_escape(ds)}}}")
    header1.append(rf"\multicolumn{{{n_metrics}}}{{c}}{{Average}}")
    lines.append("    " + " & ".join(header1) + r" \\")
    
    # Header row 2: I2T T2I [Grp] for each
    header2 = ["Model"]
    metric_labels = ["I2T", "T2I", "Grp"] if has_group else ["I2T", "T2I"]
    for ds in valid_datasets:
        header2.extend(metric_labels)
    header2.extend(metric_labels)  # For average
    lines.append("    " + " & ".join(header2) + r" \\")
    lines.append(r"    \midrule")
    
    # Compute averages
    avg_i2t = pivot_i2t[valid_datasets].mean(axis=1)
    avg_t2i = pivot_t2i[valid_datasets].mean(axis=1)
    avg_grp = pivot_grp[valid_datasets].mean(axis=1) if has_group else None
    
    best_avg_i2t = avg_i2t.loc[ordered_methods].idxmax()
    best_avg_t2i = avg_t2i.loc[ordered_methods].idxmax()
    best_avg_grp = avg_grp.loc[ordered_methods].idxmax() if has_group else None
    
    # Body rows
    for i, method in enumerate(ordered_methods):
        if i in split_idxs:
            lines.append(r"    \midrule")
        
        row = [latex_escape(method)]
        
        for ds in valid_datasets:
            # I2T
            val_i2t = fmt(pivot_i2t.loc[method, ds]) if ds in pivot_i2t.columns else "--"
            val_i2t = style_cell(val_i2t, method, method == best_i2t.get(ds))
            row.append(val_i2t)
            
            # T2I
            val_t2i = fmt(pivot_t2i.loc[method, ds]) if ds in pivot_t2i.columns else "--"
            val_t2i = style_cell(val_t2i, method, method == best_t2i.get(ds))
            row.append(val_t2i)
            
            # Group
            if has_group:
                val_grp = fmt(pivot_grp.loc[method, ds]) if ds in pivot_grp.columns else "--"
                val_grp = style_cell(val_grp, method, method == best_grp.get(ds))
                row.append(val_grp)
        
        # Averages
        val_avg_i2t = fmt(avg_i2t.loc[method])
        val_avg_i2t = style_cell(val_avg_i2t, method, method == best_avg_i2t)
        row.append(val_avg_i2t)
        
        val_avg_t2i = fmt(avg_t2i.loc[method])
        val_avg_t2i = style_cell(val_avg_t2i, method, method == best_avg_t2i)
        row.append(val_avg_t2i)
        
        if has_group:
            val_avg_grp = fmt(avg_grp.loc[method])
            val_avg_grp = style_cell(val_avg_grp, method, method == best_avg_grp)
            row.append(val_avg_grp)
        
        lines.append("    " + " & ".join(row) + r" \\")
    
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"  \end{adjustbox}")
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

# Generate bidirectional table
latex_bidirectional = make_latex_bidirectional_table(
    df,
    caption="Bidirectional performance on datasets with both image and text contrastive evaluation. I2T = Image-to-Text (text contrastive acc), T2I = Text-to-Image (image contrastive acc), Grp = Group accuracy.",
    label="tab:bidirectional",
)
print(latex_bidirectional)

Datasets with I2T/T2I: ['COCO-CF', 'COLA', 'ColorSwap', 'MMVP', 'SPEC', 'VisMin', 'Winoground']
\begin{table}[t]
  \centering
  \scriptsize
  \caption{Bidirectional performance on datasets with both image and text contrastive evaluation. I2T = Image-to-Text (text contrastive acc), T2I = Text-to-Image (image contrastive acc), Grp = Group accuracy.}
  \label{tab:bidirectional}
  \begin{adjustbox}{max width=\textwidth}
  \setlength{\tabcolsep}{3pt}
  \begin{tabular}{l|ccc|ccc|ccc|ccc|ccc|ccc|ccc|ccc}
    \toprule
     & \multicolumn{3}{c}{COCO-CF} & \multicolumn{3}{c}{COLA} & \multicolumn{3}{c}{ColorSwap} & \multicolumn{3}{c}{MMVP} & \multicolumn{3}{c}{SPEC} & \multicolumn{3}{c}{VisMin} & \multicolumn{3}{c}{Winoground} & \multicolumn{3}{c}{Average} \\
    Model & I2T & T2I & Grp & I2T & T2I & Grp & I2T & T2I & Grp & I2T & T2I & Grp & I2T & T2I & Grp & I2T & T2I & Grp & I2T & T2I & Grp & I2T & T2I & Grp \\
    \midrule
    CLIP & \underline{74.0} & \underline{73.1} & \underline{60.4} & \un

## 7. Per-Dataset Bidirectional Tables (I2T / T2I / Group by Subset)

In [50]:
def make_latex_bidirectional_per_dataset_table(
    df, dataset_name,
    caption=None,
    label=None,
    decimals=1,
    font_size="scriptsize",
    tabcolsep="4pt"
):
    """
    Create a per-dataset LaTeX table showing I2T, T2I, and Group accuracy for each subset.
    Columns: Model | Subset1 (I2T T2I Grp) | Subset2 (I2T T2I Grp) | ... | Avg (I2T T2I Grp)
    """
    from evalviz.tables import _run_info_from_df, order_rows_by_groups, latex_escape, _ensure_method_flags
    
    df = _ensure_method_flags(df)
    
    # Filter to this dataset
    ds_df = df[df["dataset"] == dataset_name].copy()
    if ds_df.empty:
        return f"% No data for dataset: {dataset_name}\n"
    
    # Check available metrics
    available_metrics = ds_df["metric"].unique()
    has_i2t = "text_contrastive_accuracy" in available_metrics
    has_t2i = "image_contrastive_accuracy" in available_metrics
    has_grp = "group_contrastive_accuracy" in available_metrics
    
    if not (has_i2t and has_t2i):
        return f"% Dataset {dataset_name} doesn't have both I2T and T2I metrics.\n"
    
    # Get unique subsets
    subsets = sorted(ds_df["subset"].unique())
    
    # Check if this dataset has meaningful subsets (more than just 'all')
    meaningful_subsets = [s for s in subsets if s.lower() != "all"]
    has_subsets = len(meaningful_subsets) > 0
    
    if not has_subsets:
        # Dataset doesn't have subsets - return simple table without subset breakdown
        return f"% Dataset {dataset_name} has no subsets (only 'all'). Use the overall bidirectional table instead.\n"
    
    # Build pivot tables for each metric
    def build_subset_pivot(metric_name):
        d = ds_df[ds_df["metric"] == metric_name].copy()
        if d.empty:
            return pd.DataFrame()
        # Average over seeds
        agg = d.groupby(["subset", "run_label"], as_index=False)["value"].mean()
        return agg.pivot_table(index="run_label", columns="subset", values="value")
    
    pivot_i2t = build_subset_pivot("text_contrastive_accuracy")
    pivot_t2i = build_subset_pivot("image_contrastive_accuracy")
    pivot_grp = build_subset_pivot("group_contrastive_accuracy") if has_grp else pd.DataFrame()
    
    # Get common models
    all_models = set(pivot_i2t.index) & set(pivot_t2i.index)
    if has_grp and not pivot_grp.empty:
        all_models = all_models & set(pivot_grp.index)
    all_models = sorted(all_models)
    
    # Filter subsets to those in our pivots - exclude 'all' since we want per-subset breakdown
    valid_subsets = [s for s in subsets 
                     if s.lower() != "all" 
                     and s in pivot_i2t.columns 
                     and s in pivot_t2i.columns]
    
    # Also check group pivot if it exists
    if has_grp and not pivot_grp.empty:
        valid_subsets = [s for s in valid_subsets if s in pivot_grp.columns]
    
    if not valid_subsets:
        return f"% No valid subsets for dataset: {dataset_name}\n"
    
    # Get run info for ordering
    run_info = _run_info_from_df(df)
    ordered_methods, split_idxs = order_rows_by_groups(all_models, run_info)
    
    # Helpers
    def fmt(v):
        if pd.isna(v):
            return "--"
        return f"{v * 100:.{decimals}f}"
    
    def style_cell(s, method, is_best):
        info = run_info.get(method, {})
        if info.get("is_baseline", False):
            s = r"\underline{" + s + "}"
        if info.get("is_ours", False):
            s = r"\emph{" + s + "}"
        if is_best:
            s = r"\textbf{" + s + "}"
        return s
    
    # Find best per column
    best_i2t = {}
    best_t2i = {}
    best_grp = {}
    for s in valid_subsets:
        if s in pivot_i2t.columns:
            best_i2t[s] = pivot_i2t[s].loc[ordered_methods].idxmax()
        if s in pivot_t2i.columns:
            best_t2i[s] = pivot_t2i[s].loc[ordered_methods].idxmax()
        if has_grp and s in pivot_grp.columns:
            best_grp[s] = pivot_grp[s].loc[ordered_methods].idxmax()
    
    # Compute averages
    avg_i2t = pivot_i2t[valid_subsets].mean(axis=1)
    avg_t2i = pivot_t2i[valid_subsets].mean(axis=1)
    avg_grp = pivot_grp[valid_subsets].mean(axis=1) if has_grp and not pivot_grp.empty else None
    
    best_avg_i2t = avg_i2t.loc[ordered_methods].idxmax()
    best_avg_t2i = avg_t2i.loc[ordered_methods].idxmax()
    best_avg_grp = avg_grp.loc[ordered_methods].idxmax() if avg_grp is not None else None
    
    # Column spec
    n_metrics = 3 if has_grp else 2
    col_parts = ["l"]  # Model column
    for s in valid_subsets:
        col_parts.append("|")
        col_parts.append("c" * n_metrics)
    col_parts.append("|" + "c" * n_metrics)  # Average columns
    col_spec = "".join(col_parts)
    
    # Caption/label defaults
    if caption is None:
        caption = f"{dataset_name}: Bidirectional performance by subset. I2T = text contrastive, T2I = image contrastive, Grp = group accuracy."
    if label is None:
        label = f"tab:{dataset_name.lower().replace(' ', '_').replace('-', '_')}_bidirectional"
    
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(rf"  \{font_size}")
    lines.append(f"  \\caption{{{caption}}}")
    lines.append(f"  \\label{{{label}}}")
    lines.append(r"  \begin{adjustbox}{max width=\textwidth}")
    lines.append(rf"  \setlength{{\tabcolsep}}{{{tabcolsep}}}")
    lines.append(rf"  \begin{{tabular}}{{{col_spec}}}")
    lines.append(r"    \toprule")
    
    # Header row 1: Subset names with multicolumn
    header1 = [""]
    for s in valid_subsets:
        header1.append(rf"\multicolumn{{{n_metrics}}}{{c}}{{{latex_escape(s)}}}")
    header1.append(rf"\multicolumn{{{n_metrics}}}{{c}}{{Average}}")
    lines.append("    " + " & ".join(header1) + r" \\")
    
    # Header row 2: I2T T2I [Grp] for each
    header2 = ["Model"]
    metric_labels = ["I2T", "T2I", "Grp"] if has_grp else ["I2T", "T2I"]
    for s in valid_subsets:
        header2.extend(metric_labels)
    header2.extend(metric_labels)  # For average
    lines.append("    " + " & ".join(header2) + r" \\")
    lines.append(r"    \midrule")
    
    # Body rows
    for i, method in enumerate(ordered_methods):
        if i in split_idxs:
            lines.append(r"    \midrule")
        
        row = [latex_escape(method)]
        
        for s in valid_subsets:
            # I2T
            val_i2t = fmt(pivot_i2t.loc[method, s]) if s in pivot_i2t.columns else "--"
            val_i2t = style_cell(val_i2t, method, method == best_i2t.get(s))
            row.append(val_i2t)
            
            # T2I
            val_t2i = fmt(pivot_t2i.loc[method, s]) if s in pivot_t2i.columns else "--"
            val_t2i = style_cell(val_t2i, method, method == best_t2i.get(s))
            row.append(val_t2i)
            
            # Group
            if has_grp:
                val_grp = fmt(pivot_grp.loc[method, s]) if s in pivot_grp.columns else "--"
                val_grp = style_cell(val_grp, method, method == best_grp.get(s))
                row.append(val_grp)
        
        # Averages
        val_avg_i2t = fmt(avg_i2t.loc[method])
        val_avg_i2t = style_cell(val_avg_i2t, method, method == best_avg_i2t)
        row.append(val_avg_i2t)
        
        val_avg_t2i = fmt(avg_t2i.loc[method])
        val_avg_t2i = style_cell(val_avg_t2i, method, method == best_avg_t2i)
        row.append(val_avg_t2i)
        
        if has_grp:
            val_avg_grp = fmt(avg_grp.loc[method])
            val_avg_grp = style_cell(val_avg_grp, method, method == best_avg_grp)
            row.append(val_avg_grp)
        
        lines.append("    " + " & ".join(row) + r" \\")
    
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"  \end{adjustbox}")
    lines.append(r"\end{table}")
    
    return "\n".join(lines)


# Generate per-dataset bidirectional tables
bidirectional_datasets = ["COCO-CF", "COLA", "ColorSwap", "MMVP", "SPEC", "VisMin", "Winoground"]

all_bidirectional_tables = {}
for ds in bidirectional_datasets:
    latex = make_latex_bidirectional_per_dataset_table(df, ds)
    all_bidirectional_tables[ds] = latex
    print(f"\n{'='*80}")
    print(f"Dataset: {ds}")
    print(f"{'='*80}")
    print(latex)


Dataset: COCO-CF
% Dataset COCO-CF has no subsets (only 'all'). Use the overall bidirectional table instead.


Dataset: COLA
\begin{table}[t]
  \centering
  \scriptsize
  \caption{COLA: Bidirectional performance by subset. I2T = text contrastive, T2I = image contrastive, Grp = group accuracy.}
  \label{tab:cola_bidirectional}
  \begin{adjustbox}{max width=\textwidth}
  \setlength{\tabcolsep}{4pt}
  \begin{tabular}{l|ccc|ccc}
    \toprule
     & \multicolumn{3}{c}{multi\_objects} & \multicolumn{3}{c}{Average} \\
    Model & I2T & T2I & Grp & I2T & T2I & Grp \\
    \midrule
    CLIP & \underline{41.9} & \underline{22.4} & \underline{14.8} & \underline{41.9} & \underline{22.4} & \underline{14.8} \\
    \midrule
    LaCLIP (CC12M) & 21.9 & 14.8 & 6.7 & 21.9 & 14.8 & 6.7 \\
    TripletCLIP (CC12M) & 24.3 & 26.2 & 11.4 & 24.3 & 26.2 & 11.4 \\
    \midrule
    CE CLIP & 24.8 & 23.3 & 11.0 & 24.8 & 23.3 & 11.0 \\
    CLIC CogVLM (LAION) & 34.3 & 24.3 & 14.3 & 34.3 & 24.3 & 14.3 \\
    CLIC Co

In [16]:
# =============================================================================
# DEBUG: Investigate SPEC discrepancy (30.1 vs 36.1)
# =============================================================================

# Filter to SPEC dataset and CS-CLIP model
spec_df = df[(df["dataset"] == "SPEC") & (df["run_label"] == "CS-CLIP")].copy()

print("="*80)
print("SPEC Data for CS-CLIP")
print("="*80)

# Show all unique subsets and metrics
print(f"\nUnique subsets: {spec_df['subset'].unique()}")
print(f"\nUnique metrics: {spec_df['metric'].unique()}")

# Show values for text_contrastive_accuracy
print("\n--- text_contrastive_accuracy by subset ---")
i2t_df = spec_df[spec_df["metric"] == "text_contrastive_accuracy"]
for _, row in i2t_df.iterrows():
    print(f"  {row['subset']}: {row['value']:.4f} ({row['value']*100:.1f}%)")

print(f"\n  Simple average: {i2t_df['value'].mean()*100:.1f}%")

# Check if there's image_contrastive_accuracy  
print("\n--- image_contrastive_accuracy by subset ---")
t2i_df = spec_df[spec_df["metric"] == "image_contrastive_accuracy"]
if len(t2i_df) > 0:
    for _, row in t2i_df.iterrows():
        print(f"  {row['subset']}: {row['value']:.4f} ({row['value']*100:.1f}%)")
    print(f"\n  Simple average: {t2i_df['value'].mean()*100:.1f}%")
else:
    print("  (No T2I data for SPEC)")

# Now let's see what build_summary_table computes
print("\n" + "="*80)
print("What build_summary_table computes for SPEC")
print("="*80)

from evalviz.tables import build_summary_table
pivot, _ = build_summary_table(df, metric="text_contrastive_accuracy")
if "SPEC" in pivot.columns:
    print(f"\nSPEC value in summary table: {pivot.loc['CS-CLIP', 'SPEC']*100:.1f}%")

# Check if there are duplicate rows (e.g., from different seeds/runs)
print("\n--- Row count per subset ---")
print(i2t_df.groupby("subset").size())

SPEC Data for CS-CLIP

Unique subsets: ['absolute_spatial' 'absolute_size' 'existence' 'count' 'relative_size'
 'relative_spatial']

Unique metrics: ['group_text_contrastive_accuracy' 'group_image_contrastive_accuracy'
 'group_group_contrastive_accuracy' 'object_contrastive_accuracy'
 'text_contrastive_accuracy' 'image_contrastive_accuracy'
 'group_contrastive_accuracy']

--- text_contrastive_accuracy by subset ---
  absolute_spatial: 0.1220 (12.2%)
  absolute_spatial: 0.1220 (12.2%)
  absolute_spatial: 0.1220 (12.2%)
  absolute_size: 0.3880 (38.8%)
  existence: 0.6870 (68.7%)
  count: 0.3447 (34.5%)
  relative_size: 0.3267 (32.7%)
  relative_spatial: 0.2950 (29.5%)

  Simple average: 30.1%

--- image_contrastive_accuracy by subset ---
  absolute_size: 0.3873 (38.7%)
  absolute_spatial: 0.1260 (12.6%)
  absolute_spatial: 0.1260 (12.6%)
  absolute_spatial: 0.1260 (12.6%)
  existence: 0.5110 (51.1%)
  count: 0.2998 (30.0%)
  relative_size: 0.3193 (31.9%)
  relative_spatial: 0.2680 (26.8%

In [17]:
# =============================================================================
# DEBUG: Where do the duplicate absolute_spatial rows come from?
# =============================================================================

spec_abs_spatial = df[(df["dataset"] == "SPEC") & 
                      (df["subset"] == "absolute_spatial") & 
                      (df["run_label"] == "CS-CLIP") &
                      (df["metric"] == "text_contrastive_accuracy")]

print("Duplicate absolute_spatial rows:")
print(spec_abs_spatial[["dataset", "subset", "metric", "value", "step"]].to_string())

# Check all SPEC subsets for duplicates
print("\n" + "="*80)
print("All SPEC subsets with row counts (for CS-CLIP, text_contrastive_accuracy):")
print("="*80)
spec_i2t = df[(df["dataset"] == "SPEC") & 
              (df["run_label"] == "CS-CLIP") &
              (df["metric"] == "text_contrastive_accuracy")]
print(spec_i2t.groupby("subset")[["value", "step"]].agg(["count", "mean"]))

# SOLUTION: The proper way to compute dataset average is to first average by subset
print("\n" + "="*80)
print("CORRECT computation (average unique subsets):")
print("="*80)
subset_avg = spec_i2t.groupby("subset")["value"].mean()
print(f"\nPer-subset averages:")
for subset, val in subset_avg.items():
    print(f"  {subset}: {val*100:.1f}%")
print(f"\nDataset average (equal weight per subset): {subset_avg.mean()*100:.1f}%")

Duplicate absolute_spatial rows:
     dataset            subset                     metric  value   step
8375    SPEC  absolute_spatial  text_contrastive_accuracy  0.122  15000
8375    SPEC  absolute_spatial  text_contrastive_accuracy  0.122  15000
8375    SPEC  absolute_spatial  text_contrastive_accuracy  0.122  15000

All SPEC subsets with row counts (for CS-CLIP, text_contrastive_accuracy):
                 value            step         
                 count      mean count     mean
subset                                         
absolute_size        1  0.388000     1  15000.0
absolute_spatial     3  0.122000     3  15000.0
count                1  0.344667     1  15000.0
existence            1  0.687000     1  15000.0
relative_size        1  0.326667     1  15000.0
relative_spatial     1  0.295000     1  15000.0

CORRECT computation (average unique subsets):

Per-subset averages:
  absolute_size: 38.8%
  absolute_spatial: 12.2%
  count: 34.5%
  existence: 68.7%
  relative_size: 32

In [18]:
# =============================================================================
# FIX: Corrected summary table with proper hierarchical averaging
# First average by subset, then by dataset (equal weight per subset)
# =============================================================================

def build_corrected_summary_table(
    df: pd.DataFrame,
    metric: str = "text_contrastive_accuracy",
    benchmark_type: str = "compositional",
):
    """
    Build summary table with CORRECT hierarchical averaging:
    1. First average within each (dataset, subset, run_label) - handles duplicates
    2. Then average across subsets within each (dataset, run_label) - equal weight per subset
    """
    from evalviz.tables import _ensure_method_flags, _run_info_from_df
    
    df = _ensure_method_flags(df)
    
    d = df[(df.get("benchmark_type", benchmark_type) == benchmark_type) & (df["metric"] == metric)].copy() \
        if "benchmark_type" in df.columns else df[df["metric"] == metric].copy()
    
    if d.empty:
        return pd.DataFrame(), {}
    
    run_info = _run_info_from_df(d)
    
    # CORRECT: First average by subset (handles duplicates), then by dataset
    subset_avg = d.groupby(["dataset", "subset", "run_label"], as_index=False)["value"].mean()
    dataset_avg = subset_avg.groupby(["dataset", "run_label"], as_index=False)["value"].mean()
    
    pivot = dataset_avg.pivot_table(index="run_label", columns="dataset", values="value")
    pivot = pivot[sorted(pivot.columns)]
    
    return pivot, run_info


# Compare old vs new
from evalviz.tables import build_summary_table

pivot_old, _ = build_summary_table(df, metric="text_contrastive_accuracy")
pivot_new, _ = build_corrected_summary_table(df, metric="text_contrastive_accuracy")

print("="*80)
print("COMPARISON: Old vs Corrected Summary Table (CS-CLIP)")
print("="*80)

comparison = pd.DataFrame({
    "Old (wrong)": pivot_old.loc["CS-CLIP"] * 100,
    "New (correct)": pivot_new.loc["CS-CLIP"] * 100,
    "Difference": (pivot_new.loc["CS-CLIP"] - pivot_old.loc["CS-CLIP"]) * 100
})
print(comparison.round(1).to_string())

print(f"\nOld Overall Avg: {(pivot_old.loc['CS-CLIP'].mean() * 100):.1f}%")
print(f"New Overall Avg: {(pivot_new.loc['CS-CLIP'].mean() * 100):.1f}%")

COMPARISON: Old vs Corrected Summary Table (CS-CLIP)
                  Old (wrong)  New (correct)  Difference
dataset                                                 
ARO                      86.9           86.9         0.0
AVERAGE                   NaN            NaN         NaN
BLA                      50.2           50.2         0.0
COCO-CF                  78.2           78.2         0.0
COLA                     41.0           41.0         0.0
ColorFoil                90.5           90.5         0.0
ColorSwap                59.0           59.0         0.0
ControlledImages         43.5           43.5         0.0
MMVP                     13.3           13.3         0.0
NegBench                 32.8           32.8         0.0
SPEC                     36.1           36.1         0.0
SugarCrepe               82.2           82.2         0.0
SugarCrepe++             67.4           67.4         0.0
VALSE                    74.8           74.8         0.0
VL_CheckList             79.2      